# Data Cleaning & Preprocessing
Title: Economic Development through Demographic and Social Transition in East Asia and the Pacific

This notebook merges and cleans the two raw CSV files extracted from the World Bank WDI:
- `dv_metadata.csv`: 11 indicators with full precision (main dataset)
- `dv_dataset.csv`: 6 indicators including Population Total (supplement)

The output is a single clean CSV/JSON ready for D3.js visualization.

TODO (will remove when finish everything for proper readibility)
- [✓] Load in datasets 
- [✓] Delete the footer inside the dataset
- [✓] Add population column in metadata set
- [✓] Reshape and pivot indicators / years to be in separate columns
- [✓] Add regions / subgroups 
- [✓] Analyse / handle missing data 
- [✓] Combine the clean data into proper format 
- [✓] Standardise datas / renaming them  
- [✓] Save clean data inside json and csv 
- [  ] Add additional charts / graphs for data analysation (optional, will update later)
- [  ] Clean up code structure in data cleaning - ie combine certain stuff / delete certain stuff for presentation 

Can add anything else for preprocessing as long as its relevant.

In [ ]:
import pandas as pd
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Load Raw Data

In [ ]:
raw_dir = os.path.join('..', 'raw_data')

# Load metadata file (11 indicators, full precision)
df_meta = pd.read_csv(
    os.path.join(raw_dir, 'dv_metadata.csv'),
    na_values=['..'],
    keep_default_na=True,
    encoding='cp1252'
)

# Load dataset file (6 indicators including Population Total)
df_data = pd.read_csv(
    os.path.join(raw_dir, 'dv_dataset.csv'),
    na_values=['..'],
    keep_default_na=True,
    encoding='cp1252'
)

print(f'Metadata file shape: {df_meta.shape}')
print(f'Dataset file shape:  {df_data.shape}')
print()
print('Metadata indicators:')
print(df_meta["Series Name"].dropna().unique())
print()
print('Dataset indicators:')
print(df_data["Series Name"].dropna().unique())

## 2. Clean Footer Rows
Both files have empty rows and World Bank attribution text at the bottom. Remove any row where Country Name is empty.

In [ ]:
# Drop rows where Country Name is NaN (footer/metadata rows)
df_meta = df_meta.dropna(subset=['Country Name']).copy()
df_data = df_data.dropna(subset=['Country Name']).copy()

print(f'Metadata after cleaning: {df_meta.shape}')
print(f'Dataset after cleaning:  {df_data.shape}')
print()
print(f'Countries in metadata: {df_meta["Country Name"].nunique()}')
print(f'Countries in dataset:  {df_data["Country Name"].nunique()}')

## 3. Extract Population Total from Dataset File
The metadata file has 11 indicators but is missing `Population, total`. We extract it from the dataset file and add it.

In [ ]:
# Extract Population Total rows from the dataset file
pop_total = df_data[df_data['Series Code'] == 'SP.POP.TOTL'].copy()
print(f'Population Total rows: {len(pop_total)}')
print(pop_total[['Country Name', 'Series Name']].head())

In [ ]:
# Combine: metadata (11 indicators) + Population Total (12th indicator)
df_combined = pd.concat([df_meta, pop_total], ignore_index=True)

# Remove any rows with missing Series Name (leftover from footer cleaning)
df_combined = df_combined.dropna(subset=['Series Name', 'Series Code'])

print(f'Combined shape: {df_combined.shape}')
print(f'\nAll indicators ({df_combined["Series Name"].nunique()}):')
for name in df_combined['Series Name'].unique():
    print(f'  - {name}')

## 4. Reshape: Wide to Long Format
Convert year columns (`2005 [YR2005]` ... `2024 [YR2024]`) into rows.

In [ ]:
# Identify year columns
year_cols = [c for c in df_combined.columns if c.startswith('20') and '[YR' in c]
print(f'Year columns ({len(year_cols)}): {year_cols}')

# Melt: wide -> long
df_long = df_combined.melt(
    id_vars=['Country Name', 'Country Code', 'Series Name', 'Series Code'],
    value_vars=year_cols,
    var_name='Year_raw',
    value_name='Value'
)

# Extract numeric year
df_long['Year'] = df_long['Year_raw'].str.extract(r'(\d{4})').astype(int)
df_long = df_long.drop(columns=['Year_raw'])

print(f'Long format shape: {df_long.shape}')
df_long.head()

## 5. Pivot Indicators to Columns
Each indicator becomes its own column, giving one row per country-year.

In [ ]:
# Create short column names for each indicator
indicator_names = {
    'NY.GDP.PCAP.CD': 'gdp_per_capita',
    'SP.DYN.LE00.IN': 'life_expectancy',
    'SP.DYN.CBRT.IN': 'birth_rate',
    'SP.POP.GROW': 'population_growth',
    'SP.DYN.TFRT.IN': 'fertility_rate',
    'SP.URB.TOTL.IN.ZS': 'urban_population_pct',
    'SE.SEC.ENRR': 'school_enrollment_secondary',
    'SE.PRM.ENRR': 'school_enrollment_primary',
    'SH.XPD.CHEX.PC.CD': 'health_expenditure_per_capita',
    'SL.TLF.CACT.ZS': 'labor_force_participation',
    'NE.EXP.GNFS.ZS': 'exports_pct_gdp',
    'SP.POP.TOTL': 'population_total'
}

# Map Series Code to short name
df_long['Indicator'] = df_long['Series Code'].map(indicator_names)

# Pivot
df_pivot = df_long.pivot_table(
    index=['Country Name', 'Country Code', 'Year'],
    columns='Indicator',
    values='Value',
    aggfunc='first'
).reset_index()

# Flatten column names
df_pivot.columns.name = None

# Convert indicator columns to numeric
for col in indicator_names.values():
    if col in df_pivot.columns:
        df_pivot[col] = pd.to_numeric(df_pivot[col], errors='coerce')

print(f'Pivoted shape: {df_pivot.shape}')
df_pivot.head()

## 6. Add Region Sub-Groups
Add a categorical region classification for geographical/hierarchical structure (required by the project brief).

In [ ]:
# Sub-regional groupings within East Asia & Pacific
region_map = {
    # East Asia
    'China': 'East Asia',
    'Hong Kong SAR, China': 'East Asia',
    'Macao SAR, China': 'East Asia',
    'Japan': 'East Asia',
    'Korea, Rep.': 'East Asia',
    "Korea, Dem. People's Rep.": 'East Asia',
    'Mongolia': 'East Asia',
    # Southeast Asia
    'Brunei Darussalam': 'Southeast Asia',
    'Cambodia': 'Southeast Asia',
    'Indonesia': 'Southeast Asia',
    'Lao PDR': 'Southeast Asia',
    'Malaysia': 'Southeast Asia',
    'Myanmar': 'Southeast Asia',
    'Philippines': 'Southeast Asia',
    'Singapore': 'Southeast Asia',
    'Thailand': 'Southeast Asia',
    'Timor-Leste': 'Southeast Asia',
    'Viet Nam': 'Southeast Asia',
    # Pacific Islands
    'Fiji': 'Pacific',
    'Kiribati': 'Pacific',
    'Marshall Islands': 'Pacific',
    'Micronesia, Fed. Sts.': 'Pacific',
    'Nauru': 'Pacific',
    'Palau': 'Pacific',
    'Papua New Guinea': 'Pacific',
    'Samoa': 'Pacific',
    'Solomon Islands': 'Pacific',
    'Tonga': 'Pacific',
    'Tuvalu': 'Pacific',
    'Vanuatu': 'Pacific',
    'American Samoa': 'Pacific',
    'French Polynesia': 'Pacific',
    'Guam': 'Pacific',
    'New Caledonia': 'Pacific',
    'Northern Mariana Islands': 'Pacific',
    'Australia': 'Pacific',
    'New Zealand': 'Pacific'
}

df_pivot['Region'] = df_pivot['Country Name'].map(region_map)

# Check for unmapped countries
unmapped = df_pivot[df_pivot['Region'].isna()]['Country Name'].unique()
if len(unmapped) > 0:
    print(f'WARNING: Unmapped countries: {unmapped}')
else:
    print('All countries mapped to a region.')

print(f'\nRegion distribution:')
print(df_pivot.groupby('Region')['Country Name'].nunique())

## 7. Standardise Country Names
Some countries have special characters ("-", ",", ".") or abbreviated/formal names. Replaced them with cleaner and  consistent names.

In [ ]:
# Standardise country names
country_name_map = {
    'Viet Nam': 'Vietnam',
    'Timor-Leste': 'Timor Leste',
    'Micronesia, Fed. Sts.': 'Micronesia',
    'Korea, Rep.': 'South Korea',
    "Korea, Dem. People's Rep.": 'North Korea',
    'Hong Kong SAR, China': 'Hong Kong',
    'Macao SAR, China': 'Macao',
    'Brunei Darussalam': 'Brunei',
    'Lao PDR': 'Laos'
}

df_pivot['Country Name'] = df_pivot['Country Name'].replace(country_name_map)

print('Renamed countries:')
for old, new in country_name_map.items():
    print(f'  "{old}" -> "{new}"')
print(f'\nAll country names:')
for name in sorted(df_pivot['Country Name'].unique()):
    print(f'  {name}')

## 8. Analyse Missing Data
Check which countries and indicators have the most missing values and negative values.

In [ ]:
numeric_cols = list(indicator_names.values())

# Missing values by indicator
print('=== Missing values by indicator ===')
total_rows = len(df_pivot)
for col in numeric_cols:
    missing = df_pivot[col].isna().sum()
    pct = 100 * missing / total_rows
    print(f'  {col:40s}: {missing:4d} / {total_rows} ({pct:.1f}%)')

print(f'\n=== Missing values by country (across all indicators) ===')
country_missing = df_pivot.groupby('Country Name')[numeric_cols].apply(
    lambda x: x.isna().sum().sum()
).sort_values(ascending=False)

total_possible = len(year_cols) * len(numeric_cols)
for country, missing in country_missing.items():
    pct = 100 * missing / total_possible
    print(f'  {country:35s}: {missing:4d} / {total_possible} ({pct:.1f}%)')

 # check which columns have negative values
for col in numeric_cols:
    neg_count = (df_pivot[col] < 0).sum()
    if neg_count > 0:
        print(f'\n  {col}: {neg_count} negative values')

## 9. Handle Missing Data
- **Drop countries**: Countries with more than 40% null values across all indicators are dropped (e.g North Korea with 41.7% missing and no GDP data at all).
- **Drop year 2024**: 2024 has ~48% missing values as many indicators haven't been published yet.
- **Keep remaining missing values as null**: D3.js can handle null values properly.

In [ ]:
# Missing values by year
print('=== Missing values by year ===')
for year in sorted(df_pivot['Year'].unique()):
    year_data = df_pivot[df_pivot['Year'] == year]
    total = year_data[numeric_cols].size
    missing = year_data[numeric_cols].isna().sum().sum()
    pct = 100 * missing / total
    print(f'  {year}: {missing} / {total} ({pct:.1f}%)')

# Drop 2024 (48% missing - too incomplete)
df_pivot = df_pivot[df_pivot['Year'] != 2024]
print(f'\nDropped year 2024. Remaining years: {df_pivot["Year"].min()} - {df_pivot["Year"].max()}')
print(f'Shape: {df_pivot.shape}')

In [ ]:
# Calculate missing percentage per country
country_missing_pct = df_pivot.groupby('Country Name')[numeric_cols].apply(
    lambda x: x.isna().sum().sum() / x.size * 100
)

# Countries to drop (>40% missing)
drop_countries = country_missing_pct[country_missing_pct > 40].index.tolist()
print(f'Countries to DROP (>40% missing): {drop_countries}')
print()

# Countries to keep
keep_countries = country_missing_pct[country_missing_pct <= 40].index.tolist()
print(f'Countries to KEEP ({len(keep_countries)}): {keep_countries}')

# Create df_clean (df_pivot already has renamed countries + no 2024)
df_clean = df_pivot[df_pivot['Country Name'].isin(keep_countries)].copy()
print(f'\nCleaned shape: {df_clean.shape}')

In [ ]:
# Calculate missing % by region and indicator
missing_by_region = df_clean.groupby('Region')[numeric_cols].apply(
    lambda x: x.isna().sum() / len(x) * 100
)

# Rename columns for cleaner labels
label_map = {
    'gdp_per_capita': 'GDP/Capita',
    'life_expectancy': 'Life Exp.',
    'birth_rate': 'Birth Rate',
    'fertility_rate': 'Fertility',
    'population_total': 'Population',
    'population_growth': 'Pop. Growth',
    'urban_population_pct': 'Urban %',
    'school_enrollment_secondary': 'School (Sec)',
    'school_enrollment_primary': 'School (Pri)',
    'health_expenditure_per_capita': 'Health Exp.',
    'labor_force_participation': 'Labor Force',
    'exports_pct_gdp': 'Exports %'
}
missing_by_region = missing_by_region.rename(columns=label_map)

# Plot heatmap (transposed: indicators on y-axis, regions on x-axis)
fig, ax = plt.subplots(figsize=(6, 8))
sns.heatmap(
    missing_by_region.T,
    annot=True,
    fmt='.1f',
    cmap='ylOrRd',
    cbar_kws={'label': 'Missing %'},
    linewidths=0.5,
    ax=ax
)
ax.set_title('Missing Data by Region and Indicator (%)')
ax.set_ylabel('Indicator')
ax.set_xlabel('Region')
plt.tight_layout()
plt.show()

In [ ]:
 # Correlation matrix for all numeric indicators                                                                                                                  
corr_matrix = df_clean[numeric_cols].corr()
                                                                                                                                                                    # Rename for cleaner labels                                                                                                                                      
corr_matrix = corr_matrix.rename(index=label_map, columns=label_map)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Correlation Matrix of All Indicators')
plt.tight_layout()
plt.show()

## 10. Format & Standardise Data Types
- **Reorder columns** with Region included for geographical comparison
- **Round floats** to 2 decimal places

In [ ]:
# Reorder columns
col_order = [
    'Country Name', 'Country Code', 'Region', 'Year',
    'gdp_per_capita', 'life_expectancy', 'birth_rate',
    'fertility_rate', 'population_total', 'population_growth',
    'urban_population_pct', 'school_enrollment_secondary',
    'school_enrollment_primary', 'health_expenditure_per_capita',
    'labor_force_participation', 'exports_pct_gdp'
]

df_clean = df_clean[col_order].sort_values(['Region', 'Country Name', 'Year']).reset_index(drop=True)

# Round float columns to 2 decimal places
float_cols = df_clean.select_dtypes(include='float64').columns
df_clean[float_cols] = df_clean[float_cols].round(2)

print(f'Final dataset: {df_clean.shape[0]} rows x {df_clean.shape[1]} columns')
print(f'Countries: {df_clean["Country Name"].nunique()}')
print(f'Years: {df_clean["Year"].min()} - {df_clean["Year"].max()}')
print(f'\nColumns and types:')
print(df_clean.dtypes)

## 11. Export Processed Data
- `csv` for EDA 
- `json` for D3 visualization

Uncomment to run exporting clean data.csv 

In [ ]:
# # Export as CSV
# csv_path = 'cleaned_data.csv'
# df_clean.to_csv(csv_path, index=False)
# print(f'Saved CSV: {csv_path} ({os.path.getsize(csv_path) / 1024:.1f} KB)')

# # Export as JSON (array of objects - ideal for D3.js)
# json_path = 'cleaned_data.json'

# # Replace NaN with None for valid JSON (NaN is not valid JSON)
# records = json.loads(df_clean.to_json(orient='records'))
# for record in records:
#     for key, value in record.items():
#         if isinstance(value, float) and np.isnan(value):
#             record[key] = None

# with open(json_path, 'w') as f:
#     json.dump(records, f, indent=2)

# print(f'Saved JSON: {json_path} ({os.path.getsize(json_path) / 1024:.1f} KB)')
# print(f'\nDone! Files saved in: {os.path.abspath(".")}')

In [ ]:
# Quick statistical summary
df_clean.describe()

In [ ]:
# Preview first few records
df_clean.head(10)